<a href="https://colab.research.google.com/github/kulanigayathree/J26-IT-364/blob/model%2FOil-Spill-Detection/notebooks/oil-spill/C4_05_gradio_app_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# C4 – Step 6: Gradio Frontend (web app) 🖥️
**Karanne:**
1. Drive eke `app/` folder ekak hadanawa: models, code, sample images okkoma ekathu karala (Hugging Face ekata yawanna ready)
2. Web app eka **Colab eke run karala** test karanna **link ekak** denawa 🔗
3. *(Passe)* Hugging Face Spaces ekata deploy karanawa → **permanent link** 🌐

⏳ **Runtime → Run all** → minutes 3–5. Anthimata `https://xxxx.gradio.live` wage link ekak enawa, eka click karanna.

## 1. Setup: Drive connect + Gradio install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install gradio

Mounted at /content/drive


## 2. Kalin hadapu files okkoma thiyenawada check karanna

In [ ]:
import os, json, shutil, tarfile, joblib, pandas as pd, numpy as np, sys
DRIVE_DIR = '/content/drive/MyDrive/C4_OilSpill'
DATA_DIR, RES_DIR, MODEL_DIR = f'{DRIVE_DIR}/data', f'{DRIVE_DIR}/results', f'{DRIVE_DIR}/models'
APP_DIR = f'{DRIVE_DIR}/app'

need = [f'{DRIVE_DIR}/c4_utils.py', f'{DRIVE_DIR}/c4_features.py', f'{MODEL_DIR}/model_info.json',
        f'{DATA_DIR}/features.csv', f'{DATA_DIR}/DARTIS_2019_metadata.csv', f'{DATA_DIR}/DARTIS_2019_files.tar']
for p in need:
    print('✅' if os.path.exists(p) else '❌ NAHA', p.replace(DRIVE_DIR, ''))
info = json.load(open(f'{MODEL_DIR}/model_info.json'))
print('\nModels :', list(info['files']))
print('Best   :', info['best_model'], '| threshold', info.get('thresholds', {}).get(info['best_model'], 0.5))

✅ /c4_utils.py
✅ /c4_features.py
✅ /models/model_info.json
✅ /data/features.csv
✅ /data/DARTIS_2019_metadata.csv
✅ /data/DARTIS_2019_files.tar

Models : ['Logistic Regression', 'Random Forest', 'XGBoost', 'Random Forest (tuned)', 'XGBoost (tuned)']
Best   : XGBoost (tuned) | threshold 0.44


## 3. `app/` folder eka hadanna
Copy karanawa: code files 2, models, result charts. Hugging Face ekata yawanne me folder eka.

In [ ]:
for d in ['', 'models', 'results', 'examples']:
    os.makedirs(f'{APP_DIR}/{d}', exist_ok=True)
for fn in ['c4_utils.py', 'c4_features.py']:
    shutil.copy(f'{DRIVE_DIR}/{fn}', f'{APP_DIR}/{fn}')
for fn in list(info['files'].values()) + ['model_info.json']:
    shutil.copy(f'{MODEL_DIR}/{fn}', f'{APP_DIR}/models/{fn}')
for fn in ['04b_before_after.png', '04b_threshold_tuning.png', '04b_imbalance_methods.png', '04_feature_importance.png']:
    if os.path.exists(f'{RES_DIR}/{fn}'):
        shutil.copy(f'{RES_DIR}/{fn}', f'{APP_DIR}/results/{fn}')
print('Copied ✅'); print(sorted(os.listdir(f'{APP_DIR}/models')))

Copied ✅
['logistic_regression.pkl', 'model_info.json', 'random_forest.pkl', 'random_forest_tuned.pkl', 'sar_gate.pkl', 'xgboost.pkl', 'xgboost_tuned.pkl']


## 3b. 🚪 "Gate keeper": SAR image check eka train karanna
Webcam photo, color photo wage **SAR nowana images** dunnama model eka boru oil hoyanawa. Ehema nisa detect karanna kalin check ekak:
1. **Color check**: SAR images kalu-sudu, color photo nam reject
2. **Range check + SAR-ness check**: DARTIS images ~600k balala *"SAR muhudu image ekak kohomada penne"* (brightness, speckle noise, texture) **Isolation Forest** model ekakata igennanawa. Ehema nathi images reject.

⏳ Minute 1–2 wage (backup tar eka eka warak kiyawanawa).

In [ ]:
%%writefile /content/drive/MyDrive/C4_OilSpill/app/c4_gate.py
# c4_gate.py  –  C4 Oil Spill Detection: input check ("gate keeper")
# Upload karana image eka Sentinel-1 SAR muhudu image ekak da kiyala check karanawa.
#   1) Color check  : SAR images kalu-sudu. Color photo nam reject.
#   2) Range check  : speckle noise / texture DARTIS SAR images wala range eke da?
#   3) SAR-ness     : DARTIS images walin igena gaththa Isolation Forest eka -> SAR wage nathi images reject.
import numpy as np
import cv2

COLS = ['mean', 'std', 'cv', 'resid', 'lap', 'skew', 'dark_frac', 'texture']
COLOR_THR = 8.0      # R, G, B athara saamaanya wenasa meeta wadi nam = color photo
RANGE_COLS = ['resid', 'texture', 'lap']   # SAR speckle pennana features (range check)


def colorfulness(bgr):
    b, g, r = cv2.split(bgr.astype(np.float32))
    return float((np.abs(r - g) + np.abs(g - b) + np.abs(r - b)).mean() / 3)


def sar_stats(gray):
    """Image ekaka 'SAR wage da' kiyala balana numbers 8k (speckle noise, brightness, texture)."""
    h, w = gray.shape
    s = 512 / max(h, w)
    g = cv2.resize(gray, (max(1, int(w * s)), max(1, int(h * s))), interpolation=cv2.INTER_AREA) if s < 1 else gray
    f = g.astype(np.float32)
    valid = f > 3                              # kalu border / no-data ain karanawa
    if valid.mean() < 0.2:
        valid = np.ones_like(valid)
    v = f[valid]
    m = float(v.mean()) + 1e-3
    sd = float(v.std())
    med = cv2.medianBlur(g, 5).astype(np.float32)
    lap = float(cv2.Laplacian(f, cv2.CV_32F)[valid].var()) / (m * m)
    mu = cv2.blur(f, (7, 7))
    lstd = np.sqrt(np.maximum(cv2.blur(f * f, (7, 7)) - mu * mu, 0))
    return dict(mean=m, std=sd, cv=sd / m,
                resid=float(np.abs(f - med)[valid].mean()) / m,           # speckle noise pramanaya
                lap=float(np.log1p(lap)),                                   # podi detail / noise
                skew=float(((v - m) ** 3).mean() / (sd ** 3 + 1e-6)),
                dark_frac=float(1 - valid.mean()),
                texture=float(lstd[valid].mean() / (sd + 1e-6)))


def check(path, gate):
    """(ok, message) denawa. ok=False nam detect karanne naha."""
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        return False, 'Could not read this file as an image.'
    if colorfulness(bgr) > gate.get('color_thr', COLOR_THR):
        return False, 'This is a **colour photo**. Sentinel-1 SAR images are grayscale.'
    st = sar_stats(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY))
    for c, (lo, hi) in gate.get('ranges', {}).items():     # speckle / texture SAR range eke da?
        if not lo <= st[c] <= hi:
            return False, ("This image **doesn't look like a SAR sea image**: it has too little speckle noise / "
                           "texture compared with Sentinel-1 training images." if st[c] < lo else
                           "This image **doesn't look like a SAR sea image** (unusual noise / texture).")
    if gate.get('model') is not None:
        x = np.array([[st[c] for c in gate['cols']]])
        if gate['model'].score_samples(x)[0] < gate['thr']:
            return False, ("This image **doesn't look like a SAR sea image** (its brightness, speckle noise and "
                           "texture are different from Sentinel-1 training images).")
    return True, ''


Overwriting /content/drive/MyDrive/C4_OilSpill/app/c4_gate.py


In [ ]:
import cv2
from sklearn.ensemble import IsolationForest
sys.path.insert(0, APP_DIR)
import c4_gate

stats, colors, k = [], [], 0
with tarfile.open(f'{DATA_DIR}/DARTIS_2019_files.tar') as tf:
    for m in tf:
        if not (m.isfile() and m.name.lower().endswith('.jpg')):
            continue
        k += 1
        if k % 6:                       # hama 6weni image ekama witharak (~600k)
            continue
        buf = np.frombuffer(tf.extractfile(m).read(), np.uint8)
        bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
        if bgr is None:
            continue
        colors.append(c4_gate.colorfulness(bgr))
        stats.append(c4_gate.sar_stats(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)))
S = pd.DataFrame(stats)[c4_gate.COLS]
iso = IsolationForest(n_estimators=300, random_state=42).fit(S.values)
scores = iso.score_samples(S.values)
lo, hi = S.quantile(0.005), S.quantile(0.995)
iqr = S.quantile(0.75) - S.quantile(0.25)
ranges = {c: (float(lo[c] - 0.5 * iqr[c]), float(hi[c] + 0.5 * iqr[c])) for c in c4_gate.RANGE_COLS}
gate = dict(model=iso, cols=c4_gate.COLS, thr=float(np.percentile(scores, 0.5)) - 0.02, ranges=ranges,
            color_thr=max(c4_gate.COLOR_THR, float(np.percentile(colors, 99.9)) + 2))
joblib.dump(gate, f'{APP_DIR}/models/sar_gate.pkl')
print(f'SAR images balapu ganana: {len(S)} | color max {max(colors):.2f} -> color limit {gate["color_thr"]:.1f}')
ok = [all(r[0] <= row[c] <= r[1] for c, r in ranges.items()) and sc >= gate['thr']
      for (_, row), sc in zip(S.iterrows(), scores)]
print(f'Training SAR images walin pass wena %: {np.mean(ok)*100:.1f}%')
print('SAR range (speckle/texture):', {c: tuple(round(v, 3) for v in r) for c, r in ranges.items()})
print(S.describe().loc[['mean', 'min', 'max']].round(3))

SAR images balapu ganana: 609 | color max 0.00 -> color limit 8.0
Training SAR images walin pass wena %: 100.0%
SAR range (speckle/texture): {'resid': (0.02, 0.122), 'texture': (0.008, 1.103), 'lap': (-0.009, 0.241)}
         mean     std     cv  resid    lap   skew  dark_frac  texture
mean  137.271  23.793  0.183  0.081  0.120  0.062        0.0    0.693
min    53.004   7.135  0.055  0.026  0.016 -2.609        0.0    0.132
max   209.911  69.922  0.599  0.115  0.224  3.563        0.0    0.955


## 4. Sample images thoranna (test set eken)
App eke click karanna puluwan **sample images 7k**: oil (muhuda 3, werala 1) + no-oil (muhuda 2, werala 1).
Hama ekakma **test set eken** (model eka kawadawath nodakapu ewa). Best model eka hondata allana ewa thoranawa.

In [ ]:
sys.path.insert(0, DRIVE_DIR)
from c4_features import FEATURES
feat = pd.read_csv(f'{DATA_DIR}/features.csv')
test = feat[feat.split == 'test'].copy()
best = info['best_model']
thr = info.get('thresholds', {}).get(best, 0.5)
test['pred'] = (joblib.load(f'{MODEL_DIR}/{info["files"][best]}').predict_proba(test[FEATURES])[:, 1] >= thr).astype(int)
g = test.groupby(['image', 'subset']).apply(lambda d: pd.Series(dict(
        n=len(d), tp=int(((d.pred == 1) & (d.y == 1)).sum()), fp=int(((d.pred == 1) & (d.y == 0)).sum()),
        fn=int(((d.pred == 0) & (d.y == 1)).sum())))).reset_index()
g['score'] = g.tp - 2 * g.fp - g.fn

def pick(subset, k, oil):
    s = g[g.subset == subset]
    if oil:
        c = s[s.tp > 0].sort_values('score', ascending=False)
    else:
        c = s[(s.fp == 0) & (s.n >= 3) & (s.n <= 40)].sort_values('n', ascending=False)
        c = c if len(c) else s.sort_values('fp')
    return c.head(k)

chosen = pd.concat([pick('ow', 3, True), pick('oc', 1, True), pick('nw', 2, False), pick('nc', 1, False)])
print(chosen[['image', 'subset', 'n', 'tp', 'fp', 'fn']].to_string(index=False))

                image subset  n  tp  fp  fn
          ow-0099.jpg     ow  9   9   0   0
          ow-0037.jpg     ow  8   8   0   0
          ow-0621.jpg     ow  5   5   0   0
          oc-0029.jpg     oc 10  10   0   0
nw-1780-11-000014.jpg     nw 40   0   0   0
nw-1776-11-000010.jpg     nw 40   0   0   0
nc-0143-01-000067.jpg     nc 35   0   0   0


/tmp/ipykernel_2548/136315694.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  g = test.groupby(['image', 'subset']).apply(lambda d: pd.Series(dict(


In [ ]:
meta = pd.read_csv(f'{DATA_DIR}/DARTIS_2019_metadata.csv')
want = set(chosen.image)
with tarfile.open(f'{DATA_DIR}/DARTIS_2019_files.tar') as tf:      # backup eken me images 7 witharak gannawa
    for m in tf:
        b = os.path.basename(m.name)
        if m.isfile() and b in want:
            with open(f'{APP_DIR}/examples/{b}', 'wb') as f:
                f.write(tf.extractfile(m).read())
ex = []
for _, r in chosen.iterrows():
    rows = meta[(meta.jpg_file == r.image) & meta.is_oil]
    boxes = rows[['obj_patchloc_xmin', 'obj_patchloc_ymin', 'obj_patchloc_xmax', 'obj_patchloc_ymax']].values.tolist()
    ex.append(dict(file=r.image, is_coast=r.subset in ('oc', 'nc'), boxes=boxes))
json.dump(ex, open(f'{APP_DIR}/examples/examples.json', 'w'), indent=1)
print('Sample images ✅', sorted(os.listdir(f'{APP_DIR}/examples')))

Sample images ✅ ['examples.json', 'nc-0143-01-000067.jpg', 'nw-1776-11-000010.jpg', 'nw-1780-11-000014.jpg', 'oc-0029.jpg', 'ow-0037.jpg', 'ow-0099.jpg', 'ow-0621.jpg']


## 5. Web app code eka (`app.py`) save karanna

In [ ]:
%%writefile /content/drive/MyDrive/C4_OilSpill/app/app.py
# app.py  –  C4 Oil Spill Detection: Gradio web app
# Colab eke test karanna saha Hugging Face Spaces eke host karanna ekama file eka.
import os, json, time
import numpy as np
import pandas as pd
import cv2
import joblib
import gradio as gr

import c4_utils as u
import c4_features as cf
import c4_gate
from c4_features import FEATURES

BASE = os.path.dirname(os.path.abspath(__file__))
MODEL_DIR = os.path.join(BASE, 'models')
EX_DIR = os.path.join(BASE, 'examples')
RES_DIR = os.path.join(BASE, 'results')

# ---------- Models + settings load karanawa ----------
with open(os.path.join(MODEL_DIR, 'model_info.json')) as f:
    INFO = json.load(f)
MODELS = {name: joblib.load(os.path.join(MODEL_DIR, fn))
          for name, fn in INFO['files'].items() if os.path.exists(os.path.join(MODEL_DIR, fn))}
THRESHOLDS = {name: float(INFO.get('thresholds', {}).get(name, INFO.get('threshold', 0.5))) for name in MODELS}
BEST = INFO.get('best_model') if INFO.get('best_model') in MODELS else list(MODELS)[0]
ORDER = [BEST] + [m for m in MODELS if m != BEST]
_gate_path = os.path.join(MODEL_DIR, 'sar_gate.pkl')
GATE = joblib.load(_gate_path) if os.path.exists(_gate_path) else {'model': None, 'color_thr': c4_gate.COLOR_THR}

def _thumb(path):
    """Podi 32x32 fingerprint ekak (Gradio image eka aye save kalath match wenawa)."""
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return None if img is None else cv2.resize(img, (32, 32), interpolation=cv2.INTER_AREA).astype(np.float32)


def _match_example(path):
    """Sample image ekak nam ehi ground-truth boxes (list) denawa, nathnam None."""
    t = _thumb(path)
    if t is None:
        return None
    for ref, boxes in GT_BOXES:
        if ref.shape == t.shape and np.abs(ref - t).mean() < 4:
            return boxes
    return None


GT_BOXES, EXAMPLES = [], []
if os.path.exists(os.path.join(EX_DIR, 'examples.json')):
    with open(os.path.join(EX_DIR, 'examples.json')) as f:
        ex = json.load(f)
    for e in ex:
        p = os.path.join(EX_DIR, e['file'])
        if os.path.exists(p):
            EXAMPLES.append([p, bool(e['is_coast'])])
            GT_BOXES.append((_thumb(p), e['boxes']))

OIL_RGB, LOOK_RGB, GT_RGB = (235, 60, 40), (250, 200, 0), (0, 210, 255)
EMPTY = pd.DataFrame(columns=['#', 'Class', 'Oil probability (%)', 'Area (px)', 'Elongation', 'Contrast'])


def _outlined_text(img, text, org, scale, thick):
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thick + 2, cv2.LINE_AA)
    cv2.putText(img, text, org, cv2.FONT_HERSHEY_SIMPLEX, scale, (255, 255, 255), thick, cv2.LINE_AA)


def analyze(path, model_name, threshold, is_coast, skip_check=False):
    """Image eka -> (SAR check) -> dark patches -> features -> ML model -> oil / look-alike."""
    if path is None:
        return None, '⬅️ Upload a Sentinel-1 SAR image or click a sample image below.', EMPTY
    t0 = time.time()
    example_boxes = _match_example(path)
    if not skip_check and example_boxes is None:      # sample images DARTIS eken, check oni naha
        ok, why = c4_gate.check(path, GATE)
        if not ok:
            bgr = cv2.imread(path, cv2.IMREAD_COLOR)
            shown = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) if bgr is not None else None
            return shown, f"""## ⚠️ Not a SAR sea image: detection skipped
{why}

This model only works on **Sentinel-1 SAR images of the sea**. On other images (photos, webcam, screenshots)
any dark area would be wrongly treated as a possible oil slick, so the app refuses them.
Please upload a SAR image or click a sample image below.""", EMPTY
    res = u.detect(path)
    img = res['img']
    rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    H, W = img.shape
    lw = max(1, round(min(H, W) / 400))
    gt = example_boxes or []

    rows = cf.patch_features(res, int(is_coast))
    if not rows:
        return rgb, '🟢 **No dark patches found** in this image.', EMPTY

    X = pd.DataFrame(rows)[FEATURES]
    prob = MODELS[model_name].predict_proba(X)[:, 1]
    props = res['props']
    is_oil = prob >= threshold
    labels = res['labels']
    oil_mask = np.isin(labels, [p.label for p, o in zip(props, is_oil) if o])
    look_mask = (labels > 0) & ~oil_mask

    # ---- picture: oil = rathu fill + rathu ira, look-alike = kaha ira, ground truth = cyan kotu ----
    over = rgb.copy()
    over[oil_mask] = OIL_RGB
    out = cv2.addWeighted(over, 0.45, rgb, 0.55, 0)
    for mask, color, t in [(look_mask, LOOK_RGB, lw), (oil_mask, OIL_RGB, lw + 1)]:
        cnts, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(out, cnts, -1, color, t)
    for (x0, y0, x1, y1) in gt:
        cv2.rectangle(out, (int(x0), int(y0)), (int(x1), int(y1)), GT_RGB, lw)

    rank = np.argsort(-prob)
    for r, i in enumerate(rank, start=1):
        if is_oil[i]:
            cy, cx = props[i].centroid
            _outlined_text(out, str(r), (int(cx), int(cy)), 0.45 * lw + 0.2, lw)

    # ---- summary ----
    n_oil, n_all = int(is_oil.sum()), len(prob)
    valid = int((res['filtered'] > 0).sum()) or H * W
    oil_area = sum(p.area for p, o in zip(props, is_oil) if o) / valid * 100
    verdict = (f'## 🔴 Possible oil spill: {n_oil} oil patch{"es" if n_oil != 1 else ""} detected'
               if n_oil else '## 🟢 No oil detected: all dark patches look like look-alikes')
    summary = f"""{verdict}

| | |
|---|---|
| Dark patches found | **{n_all}** |
| 🔴 Oil | **{n_oil}** |
| 🟡 Look-alike | **{n_all - n_oil}** |
| Oil area | **{oil_area:.2f}%** of the image |
| Highest oil probability | **{prob.max() * 100:.1f}%** |
| Model / threshold | {model_name} / {threshold:.2f} |
| Processing time | {time.time() - t0:.2f} s |
""" + ('\n🟦 Cyan boxes = ground-truth oil locations from the DARTIS dataset.' if gt else '')

    table = pd.DataFrame({
        '#': np.arange(1, n_all + 1),
        'Class': ['🔴 Oil' if is_oil[i] else '🟡 Look-alike' for i in rank],
        'Oil probability (%)': np.round(prob[rank] * 100, 1),
        'Area (px)': [int(rows[i]['area']) for i in rank],
        'Elongation': [round(rows[i]['elongation'], 2) for i in rank],
        'Contrast': [round(rows[i]['contrast'], 3) for i in rank],
    }).head(25)
    return out, summary, table


def metrics_table():
    rows = []
    for r in INFO.get('metrics', []):
        if r['Model'] == 'Logistic Regression':
            rows.append(dict(Model='Logistic Regression (C4_04)', Threshold=0.5, **{k: r[k] for k in
                             ['Precision', 'Recall', 'F1', 'ROC_AUC', 'Accuracy']}))
    rows += INFO.get('tuned_metrics', []) or [dict(Threshold=0.5, **r) for r in INFO.get('metrics', [])
                                              if r['Model'] != 'Logistic Regression']
    df = pd.DataFrame(rows)
    cols = [c for c in ['Model', 'Threshold', 'Precision', 'Recall', 'F1', 'ROC_AUC', 'Accuracy'] if c in df]
    return df[cols].round(3)


# ---------- UI ----------
with gr.Blocks(title='C4 Oil Spill Detection') as demo:
    gr.Markdown("""# 🛢️ Oil Spill Detection from Sentinel-1 SAR Images
**C4: AI-Based Integrated Ocean Monitoring for Sri Lankan Waters.** Upload a SAR image: image processing finds
**dark patches**, then a machine learning model classifies each one as **🔴 oil** or **🟡 look-alike**.""")

    with gr.Tab('🔍 Detect'):
        with gr.Row():
            with gr.Column(scale=1):
                img_in = gr.Image(type='filepath', sources=['upload'], label='SAR image (Sentinel-1, grayscale)', height=360)
                model_dd = gr.Dropdown(ORDER, value=BEST, label='Model')
                thr = gr.Slider(0.05, 0.95, value=THRESHOLDS[BEST], step=0.01,
                                label='Decision threshold (oil if probability ≥ threshold)')
                coast = gr.Checkbox(value=False, label='Coastal image (land visible)')
                skip = gr.Checkbox(value=False, label='Skip SAR image check (advanced: only for SAR images from other sources)')
                btn = gr.Button('Detect oil', variant='primary')
                if EXAMPLES:
                    gr.Examples(EXAMPLES, inputs=[img_in, coast], label='Sample images (DARTIS test set)')
            with gr.Column(scale=1):
                img_out = gr.Image(label='Result: 🔴 oil  🟡 look-alike  🟦 ground truth', height=420)
                summary = gr.Markdown()
        table = gr.Dataframe(value=EMPTY, label='Detected patches (sorted by oil probability)', interactive=False)

        model_dd.change(lambda m: gr.Slider(value=THRESHOLDS[m]), model_dd, thr)
        btn.click(analyze, [img_in, model_dd, thr, coast, skip], [img_out, summary, table])

    with gr.Tab('📊 Model performance'):
        gr.Markdown(f"""### Test-set results (held-out satellite passes)
Best model: **{BEST}**, threshold **{THRESHOLDS[BEST]:.2f}**. F1 is the main metric because the data is imbalanced
(about 1 oil patch for every 5.4 look-alikes).""")
        gr.Dataframe(value=metrics_table(), interactive=False)
        for fn, cap in [('04b_before_after.png', 'Before vs after tuning, and confusion matrix of the best model'),
                        ('04b_threshold_tuning.png', 'Threshold tuning (cross-validation on training data)'),
                        ('04b_imbalance_methods.png', 'Class-imbalance handling methods (cross-validation F1)'),
                        ('04_feature_importance.png', 'Feature importance (C4_04 models)')]:
            p = os.path.join(RES_DIR, fn)
            if os.path.exists(p):
                gr.Image(p, label=cap, interactive=False)

    with gr.Tab('ℹ️ About'):
        gr.Markdown("""### How it works
1. **Speckle filtering**: median filter (5×5) reduces SAR speckle noise.
2. **Dark-patch detection**: adaptive threshold against the local sea background (bright land excluded), then morphology and connected components.
3. **Feature extraction**: 19 features per patch (shape, darkness, contrast with surrounding sea, edge sharpness, image context).
4. **Classification**: Logistic Regression, Random Forest and XGBoost (tuned) classify each patch as oil or look-alike.

5. **Input check**: before detection, the app rejects colour photos and images whose statistics (brightness, speckle noise, texture) don't match Sentinel-1 SAR sea images, using an Isolation Forest trained on DARTIS images.

### Dataset
DARTIS 2019 (Yang & Singha, 2025, PANGAEA, doi:10.1594/PANGAEA.980773, CC-BY-4.0): 3,655 Sentinel-1 images from the Mediterranean.
Train/test split by satellite pass, so test images are never seen in training. Sample images come from the test set.

### Limitations (PP1)
- Trained on Mediterranean data; Sri Lankan Sentinel-1 images will be tested in PP2.
- The dark-patch stage finds about 80% of labelled oil areas; missed areas can't be classified.
- Labels are derived automatically from bounding boxes, so some label noise is expected.
- Uses 8-bit JPG images, not calibrated SAR backscatter.
- The input check is a safety filter, not a guarantee: some non-SAR grayscale images with heavy noise may still pass.

**PP2:** U-Net deep learning segmentation (Krestenitis M4D dataset), drift modelling (OpenDrift) and AIS vessel attribution.""")

if __name__ == '__main__':
    demo.launch()


Overwriting /content/drive/MyDrive/C4_OilSpill/app/app.py


## 6. Hugging Face walata oni files 2k: `requirements.txt` + `README.md`
Colab eke thiyena **ekama library versions** liyanawa (models hariyata load wenna).

In [ ]:
import sklearn, xgboost, cv2, skimage, scipy, gradio, platform
req = f'''numpy=={np.__version__}
pandas=={pd.__version__}
scikit-learn=={sklearn.__version__}
xgboost=={xgboost.__version__}
scikit-image=={skimage.__version__}
scipy=={scipy.__version__}
joblib=={joblib.__version__}
opencv-python-headless
'''
open(f'{APP_DIR}/requirements.txt', 'w').write(req)
py = '.'.join(platform.python_version_tuple()[:2])
readme = f'''---
title: C4 Oil Spill Detection
emoji: 🛢️
colorFrom: blue
colorTo: gray
sdk: gradio
sdk_version: {gradio.__version__}
python_version: "{py}"
app_file: app.py
pinned: false
license: cc-by-4.0
---

Oil spill detection from Sentinel-1 SAR images (DARTIS 2019 dataset).
Dark-patch detection with image processing, then oil vs look-alike classification with tuned XGBoost / Random Forest.
'''
open(f'{APP_DIR}/README.md', 'w').write(readme)
print(req); print('Python', py, '| Gradio', gradio.__version__)
print('app/ folder ✅', sorted(os.listdir(APP_DIR)))

numpy==2.1.3
pandas==2.2.3
scikit-learn==1.6.1
xgboost==3.4.1
scikit-image==0.25.2
scipy==1.16.3
joblib==1.6.0
opencv-python-headless

Python 3.13 | Gradio 6.30.0
app/ folder ✅ ['.gradio', 'README.md', '__pycache__', 'app.py', 'c4_features.py', 'c4_gate.py', 'c4_utils.py', 'examples', 'models', 'requirements.txt', 'results']


## 7. 🚀 App eka run karanna (test link)
Pahala `Running on public URL: https://xxxx.gradio.live` link eka **click karanna** (new tab eke open wenawa).
- Link eka **paya 72k** witharai wada karanne, Colab tab eka open thiyenakan. (Permanent link eka Step 8 eken.)
- App eka wenas karala aye run karanna oni nam: **Runtime → Restart session** → aye **Run all**.

In [ ]:
os.chdir(APP_DIR)
sys.path.insert(0, APP_DIR)
import app
app.demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://bc01d2f5ecb7dcbd42.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 8. 🌐 (Passe) Hugging Face Spaces ekata deploy karanna: permanent link
**Mulin karanna (ekama warak):**
1. **huggingface.co** → Sign up (personal Gmail) → email eka verify karanna
2. Uda dakunu profile pic → **Settings → Access Tokens → Create new token** → Type: **Write** → nama `colab` → **Create** → token eka copy karanna (`hf_...`)

**Ita passe:** pahala cell eke `DEPLOY = False` eka **`DEPLOY = True`** karala, **ekama cell eka** run karanna (▶️). Token eka ahuwama paste karala Enter.
⏳ Build wenna minutes 3–5. Link eka: `https://huggingface.co/spaces/<oyage-username>/c4-oil-spill`

In [13]:
DEPLOY = False          # <- deploy karanna oni welawata True karanna

if DEPLOY:
    from getpass import getpass
    from huggingface_hub import HfApi
    api = HfApi(token=getpass('Hugging Face write token (hf_...): '))
    user = api.whoami()['name']
    repo = f'{user}/c4-oil-spill'
    api.create_repo(repo, repo_type='space', space_sdk='gradio', exist_ok=True)
    api.upload_folder(folder_path=APP_DIR, repo_id=repo, repo_type='space',
                      ignore_patterns=['__pycache__/*', '*.pyc'])
    print('✅ Uploaded! Build eka balanna:', f'https://huggingface.co/spaces/{repo}')
else:
    print('Deploy skip kala (DEPLOY = False).')

Deploy skip kala (DEPLOY = False).


## ✅ Iwarai!
📸 **Screenshots ewanna:** app eke **sample image ekak** click karala → **Detect oil** → result eka.

**🔔 COMMIT:** File → Save a copy in GitHub → `notebooks/oil-spill/C4_05_gradio_app.ipynb` → `feat(oil-spill): add Gradio web app for oil spill detection`